# Final Project: Deep Fake Detection

by Patrick Donnelly & Burke Havranek

EECE 5644: Introduction to Machine Learning and Pattern Recognition

Northeastern University College of Engineering

Summer 2026, Session B

## Part 1: Loading and Sanitization

### Required Imports:

In [ ]:
#!/bin/python3
# --Beginning of Code--
import sys, subprocess
def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *pkgs])

pipq("scikit-learn", "pandas", "numpy", "matplotlib", "seaborn", "pandas", "kagglehub", "ipywidgets", "parfor", "torch", "torchvision")

# Standard processing
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 80)
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os

# Useful built-ins
import shutil
from parfor import parfor
from PIL import Image

# Anticipated framework
import torch
from torchvision.transforms.functional import to_tensor

# Constants
IMAGE_SIZE = 224 # Common CV standard
TRAIN_FRAC = 0.8 # Common industry standard
SEED = 0xDEADBEEF # Humorous personal standard
DOWNSAMPLER = Image.LANCZOS # Industry standard downsampler

RAW_DIR = "images" # Determined by Kaggle
SANITIZED_DIR = "sanitized-images" # Arbitrary

REAL_DIR = "real" # Determined by Kaggle
FAKE_DIR = "fake" # Determined by Kaggle
FLUX_DIR = ["FLUX_DEV", "FLUX_PRO", "SDXL"] # Determined by Kaggle

N_REAL = 70_000
N_FAKE = [7273, 3209, 53087]

# ImageNet normalization
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

### Download and Verify Raw Data:

In [ ]:
# Check whether raw data is loaded
if not os.path.isdir(f"{RAW_DIR}"):
    print("RAW DATA DIRECTORY NOT FOUND, DOWNLOADING...", end="")
    
    # Download data
    path = kagglehub.dataset_download("shreyanshpatel1/130k-real-vs-fake-face")
    root = os.path.join(path, f"{RAW_DIR}")
    
    print(f"DONE\nCOPYING DATA FROM {root}...", end="")
    
    # Copy to local directory structure
    shutil.copytree(root, f"./{RAW_DIR}")
    
    print("DONE\nSETTING PERMISSIONS...", end="")
    
    # Change raw data to read-only for safety (parallelized)
    for root, _, files in os.walk(f"./{RAW_DIR}"):
        @parfor(files, (root,))
        def make_read_only(file, root):
            os.chmod(os.path.join(root, file), 0o444)
    
    print("DONE")
else:
    print("RAW DATA DIRECTORY FOUND, CONTINUING")

print("VERIFYING RAW DATA INTEGRITY...", end="")

try:
    # Verify raw data integrity
    assert len(os.listdir(f"./{RAW_DIR}")) == 2
    assert len(os.listdir(f"./{RAW_DIR}/{REAL_DIR}")) == N_REAL
    assert len(os.listdir(f"./{RAW_DIR}/{FAKE_DIR}")) == len(FLUX_DIR)
    for d, N in zip(FLUX_DIR, N_FAKE):
        assert len(os.listdir(f"./{RAW_DIR}/{FAKE_DIR}/{d}")) == N
except:
    raise RuntimeError("RAW DATA INTEGRITY COMPROMISED, PLEASE REMOVE AND RE-DOWNLOAD")

print("DONE\nRAW DATA SUCCESSFULLY VERIFIED")

### Create Directory Structure for Sanitized Data:

In [ ]:
if not os.path.isdir(f"{SANITIZED_DIR}"):
    print("SANITIZED DATA DIRECTORY NOT FOUND, CREATING STRUCTURE...", end="")
    
    os.mkdir(f"{SANITIZED_DIR}")
    
    # Copy directories, but not files
    for root, dirs, _ in os.walk(f"./{RAW_DIR}"):
        sanitized_root = root.replace(RAW_DIR, SANITIZED_DIR)
        for dir in dirs:
            os.mkdir(os.path.join(sanitized_root, dir))
            
    print("DONE")
else:
    print("SANITIZED DATA DIRECTORY FOUND, CONTINUING")

print("VERIFYING SANITIZED DATA DIRECTORY STRUCTURE...", end="")

try:
    # Verify sanitized data directory structure integrity
    assert len(os.listdir(f"./{SANITIZED_DIR}")) == 2
    assert len(os.listdir(f"./{SANITIZED_DIR}/{FAKE_DIR}")) == len(FLUX_DIR)
except:
    raise RuntimeError("SANITIZED DATA DIRECTORY STRUCTURE COMPROMISED, PLEASE REMOVE AND RE-RUN")

print("DONE\nSANITIZED DATA DIRECTORY STRUCTURE SUCCESSFULLY VERIFIED")

### Preprocess Images:
- Load all images and verify color encoding (RGB)
- Downsample all images to same resolution using the Lanczos algorithm
- Encode and normalize all images using PyTorch vector
- Save to same directory structure

In [ ]:
# Iterate over each image and preprocess (parallelized)
if os.path.isdir(f"{SANITIZED_DIR}") and len(os.listdir(f"./{SANITIZED_DIR}/{REAL_DIR}")) == 0:
    print("PREPROCESSING IMAGE DATA...", end="")
    
    for root, _, files in os.walk(f"./{RAW_DIR}"):
        @parfor(files, (root, RAW_DIR, SANITIZED_DIR, IMAGE_SIZE, DOWNSAMPLER, MEAN, STD))
        def preprocess_image(file, root, raw_dir, san_dir, res, alg, u, o):
            raw_path = os.path.join(root, file)
            san_path = os.path.splitext(raw_path.replace(raw_dir, san_dir))[0] + ".pt"
            
            # Load image, verify encoding, resize using Lanczos
            img = Image.open(raw_path).convert("RGB").resize((res, res), resample=alg)
            
            # Convert to PyTorch, then normalize using ImageNet standard
            ten = (to_tensor(img).float() - u) / o
            
            # Export to new directory
            torch.save(ten, san_path)
            
    print("DONE")
else:
    print("SANITIZED DATA FOUND, CONTINUING")

print("VERIFYING SANITIZED DATA INTEGRITY...", end="")

try:
    # Verify sanitized data integrity
    assert len(os.listdir(f"./{SANITIZED_DIR}")) == 2
    assert len(os.listdir(f"./{SANITIZED_DIR}/{REAL_DIR}")) == N_REAL
    assert len(os.listdir(f"./{SANITIZED_DIR}/{FAKE_DIR}")) == len(FLUX_DIR)
    for d, N in zip(FLUX_DIR, N_FAKE):
        assert len(os.listdir(f"./{SANITIZED_DIR}/{FAKE_DIR}/{d}")) == N
except:
    raise RuntimeError("SANITIZED DATA INTEGRITY COMPROMISED, PLEASE REMOVE AND RE_PROCESS")

print("DONE\nSANITIZED DATA SUCCESSFULLY VERIFIED")

### Organize Data:

Refactor structure into level directory with associated `index.csv` for data analysis.